In [26]:

import pandas as pd
from engine import load_env, create_engine_from_env
from models.manufacture_module import get_manufacturer_df, get_active_unique_manufacturers, get_manufacturer_bulletins_json, print_bulletin_details, convert_bulletin_to_df, save_vehicle_models_to_csv, batch_save_manufacturer_models, search_models_by_description
 

In [27]:

import json
from typing import Union, Any

def parse_json_string(json_string: str) -> Union[dict, list]:
    """
    Parse a JSON string into a Python object (dict or list).

    Args:
        json_string (str): A valid JSON string.

    Returns:
        dict or list: Parsed JSON object.

    Raises:
        ValueError: If the input is not valid JSON.
        TypeError: If input is not a string.
    """
    if not isinstance(json_string, str):
        raise TypeError("Input must be a JSON string")

    try:
        return json.loads(json_string)
    except json.JSONDecodeError as exc:
        raise ValueError(f"Invalid JSON string: {exc}") from exc
 

In [28]:

from pathlib import Path
from datetime import datetime

def load_latest_oem_data(manufacturer: str, landing_zone_path: str = None):
    """
    Load the latest OEM data file from the landing zone for a given manufacturer.

    Args:
        manufacturer (str): Manufacturer name (e.g., "hyundai", "mazda", "mitsubishi")
        landing_zone_path (str): Path to landing zone directory.
                                 Defaults to accy_v2/data/landing_zone relative to current dir

    Returns:
        pd.DataFrame: DataFrame loaded from the latest file for the manufacturer

    Raises:
        FileNotFoundError: If manufacturer folder or data files not found
        ValueError: If no supported files found (CSV or XLSX)
    """
    if landing_zone_path is None:
        landing_zone_path = Path("../data/landing_zone")
    else:
        landing_zone_path = Path(landing_zone_path)

    # Normalize manufacturer name (case-insensitive)
    manufacturer_clean = manufacturer.lower().strip()
    manufacturer_dir = landing_zone_path / manufacturer_clean

    # Check if manufacturer directory exists
    if not manufacturer_dir.exists():
        raise FileNotFoundError(
            f"Manufacturer folder not found: {manufacturer_dir}\n"
            f"Available manufacturers: {[d.name for d in landing_zone_path.iterdir() if d.is_dir()]}"
        )

    # Find all data files (CSV or XLSX)
    data_files = []
    for ext in ["*.csv", "*.xlsx", "*.xls"]:
        data_files.extend(manufacturer_dir.glob(ext))

    if not data_files:
        raise ValueError(
            f"No data files (CSV/XLSX) found in {manufacturer_dir}\n"
            f"Files in directory: {list(manufacturer_dir.iterdir())}"
        )

    # Get the latest file by modification time
    latest_file = max(data_files, key=lambda p: p.stat().st_mtime)
    mod_time = datetime.fromtimestamp(latest_file.stat().st_mtime)

    print(f"[OK] Loading {manufacturer.upper()} data")
    print(f"     File: {latest_file.name}")
    print(f"     Modified: {mod_time.strftime('%Y-%m-%d %H:%M:%S')}")

    # Load based on file type
    try:
        if latest_file.suffix.lower() == ".csv":
            df = pd.read_csv(latest_file)
        elif latest_file.suffix.lower() in [".xlsx", ".xls"]:
            df = pd.read_excel(latest_file)
        else:
            raise ValueError(f"Unsupported file type: {latest_file.suffix}")

        print(f"     Rows: {len(df)}, Columns: {len(df.columns)}")
        return df

    except Exception as e:
        raise RuntimeError(f"Failed to load file {latest_file.name}: {str(e)}")



In [29]:

# # Loaded environment variables and created database engine
# load_env()
# engine = create_engine_from_env()


In [30]:

# # # Batch save vehicle models for specified manufacturers
# manufactures = ["Hyundai","Honda","Kia","Mazda","Genesis", "Mitsubishi", "Volkswagen"]

# # batch_save_manufacturer_models(engine, manufactures)


In [31]:

model_number_db = pd.read_csv("db/db_vehicle_models.csv")



In [32]:
from pathlib import Path
from datetime import datetime

def load_latest_oem_data(
    manufacturer: str, 
    landing_zone_path: str = None, 
    header_row: int = None,
    auto_detect_header: bool = True,
    header_keywords: list = None
):
    """
    Load the latest OEM data file from the landing zone for a given manufacturer.

    Args:
        manufacturer (str): Manufacturer name (e.g., "hyundai", "mazda", "mitsubishi")
        landing_zone_path (str): Path to landing zone directory.
                                 Defaults to accy_v2/data/landing_zone relative to current dir
        header_row (int): Specific row index to use as headers (0-based). 
                         If None and auto_detect_header=True, will auto-detect.
                         If None and auto_detect_header=False, defaults to 0 (first row).
        auto_detect_header (bool): If True, searches for row containing header keywords.
        header_keywords (list): Keywords to search for in rows (e.g., ["Year", "Model", "Trim"]).
                               Defaults to ["Year", "Model", "Trim", "Make", "Manufacturer"]

    Returns:
        pd.DataFrame: DataFrame loaded from the latest file for the manufacturer

    Raises:
        FileNotFoundError: If manufacturer folder or data files not found
        ValueError: If no supported files found (CSV or XLSX)
    """
    if landing_zone_path is None:
        landing_zone_path = Path("../data/landing_zone")
    else:
        landing_zone_path = Path(landing_zone_path)

    # Default header keywords to search for
    if header_keywords is None:
        header_keywords = ["Year", "Model", "Trim"]

    # Normalize manufacturer name (case-insensitive)
    manufacturer_clean = manufacturer.lower().strip()
    manufacturer_dir = landing_zone_path / manufacturer_clean

    # Check if manufacturer directory exists
    if not manufacturer_dir.exists():
        raise FileNotFoundError(
            f"Manufacturer folder not found: {manufacturer_dir}\n"
            f"Available manufacturers: {[d.name for d in landing_zone_path.iterdir() if d.is_dir()]}"
        )

    # Find all data files (CSV or XLSX)
    data_files = []
    for ext in ["*.csv", "*.xlsx", "*.xls"]:
        data_files.extend(manufacturer_dir.glob(ext))

    if not data_files:
        raise ValueError(
            f"No data files (CSV/XLSX) found in {manufacturer_dir}\n"
            f"Files in directory: {list(manufacturer_dir.iterdir())}"
        )

    # Get the latest file by modification time
    latest_file = max(data_files, key=lambda p: p.stat().st_mtime)
    mod_time = datetime.fromtimestamp(latest_file.stat().st_mtime)

    print(f"[OK] Loading {manufacturer.upper()} data")
    print(f"     File: {latest_file.name}")
    print(f"     Modified: {mod_time.strftime('%Y-%m-%d %H:%M:%S')}")

    # Load based on file type
    try:
        if latest_file.suffix.lower() == ".csv":
            df_raw = pd.read_csv(latest_file, header=None)
        elif latest_file.suffix.lower() in [".xlsx", ".xls"]:
            df_raw = pd.read_excel(latest_file, header=None)
        else:
            raise ValueError(f"Unsupported file type: {latest_file.suffix}")

        # Auto-detect header row if requested
        if header_row is None and auto_detect_header:
            header_row = _find_header_row(df_raw, header_keywords)
            if header_row is None:
                print(f"     [WARNING] Could not auto-detect header row. Using row 0.")
                header_row = 0
            else:
                print(f"     [Auto-detected] Header row: {header_row}")
        elif header_row is None:
            header_row = 0
        
        # Apply header row
        if header_row > 0:
            df_raw.columns = df_raw.iloc[header_row]
            df = df_raw.iloc[header_row + 1:].reset_index(drop=True)
        else:
            df = df_raw.copy()
            if header_row == 0:
                df.columns = df.iloc[0]
                df = df.iloc[1:].reset_index(drop=True)

        print(f"     Rows: {len(df)}, Columns: {len(df.columns)}")
        print(f"     Column names: {list(df.columns[:5])}{'...' if len(df.columns) > 5 else ''}")
        return df

    except Exception as e:
        raise RuntimeError(f"Failed to load file {latest_file.name}: {str(e)}")


def _find_header_row(df: pd.DataFrame, keywords: list) -> int:
    """
    Find the row index that contains most of the header keywords.
    
    Args:
        df (pd.DataFrame): DataFrame loaded without headers (all rows are data)
        keywords (list): List of keywords to search for (e.g., ["Year", "Model", "Trim"])
    
    Returns:
        int: Row index of detected header row, or None if not found
    """
    keywords_lower = [kw.lower() for kw in keywords]
    best_row = None
    best_score = 0
    
    # Search first 10 rows for headers
    for row_idx in range(min(10, len(df))):
        row_values = [str(val).lower() for val in df.iloc[row_idx]]
        
        # Count how many keywords match in this row
        matches = sum(1 for kw in keywords_lower if any(kw in val for val in row_values))
        
        if matches > best_score:
            best_score = matches
            best_row = row_idx
    
    # Only return if we found at least 2 matching keywords
    return best_row if best_score >= 2 else None
      

In [33]:
model_number_db.info()

<class 'pandas.DataFrame'>
RangeIndex: 1168 entries, 0 to 1167
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Description   1168 non-null   str  
 1   Drivetrain    1168 non-null   str  
 2   Manufacturer  1168 non-null   str  
 3   ModelName     1168 non-null   str  
 4   ModelNumber   1168 non-null   str  
 5   ModelYear     1168 non-null   int64
 6   Package       1168 non-null   int64
 7   PassDoors     1168 non-null   int64
 8   TrimName      1140 non-null   str  
 9   engine_type   319 non-null    str  
dtypes: int64(3), str(7)
memory usage: 91.4 KB


In [34]:

# Now you can load the latest OEM data for any manufacturer
hyundai_raw_df = load_latest_oem_data("hyundai")
# print(f"\nHyundai data loaded successfully!")
# print(f"Columns: {list(hyundai_df.columns)}")
# print(f"First few rows:")
# hyundai_raw_df.head()
 

[OK] Loading HYUNDAI data
     File: 2026-8-1 HACC MAF DIST - 08102026.xlsx
     Modified: 2026-08-11 10:38:38
     [Auto-detected] Header row: 1
     Rows: 5088, Columns: 33
     Column names: ['Model Year From', 'Model Year To ', 'Model', 'Trim 1', 'Trim 2 ']...


c:\Users\paxm\AppData\Local\Programs\Python\Python314\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [35]:
# hyundai_raw_df.columns

In [36]:
# hyundai_raw_df_filtered = hyundai_raw_df[
#                                         (hyundai_raw_df['Model Year From'].notnull()) 
#                                         & (hyundai_raw_df['Model'].notnull()) 
#                                     ][["Model Year From", "Model"]]

In [37]:

# # I want to check it there any models that are in hyundai_raw_df_filtered that are not in model_number_db, and if so, I want to print them out.
# hyundai_models = set(hyundai_raw_df_filtered["Model"].str.lower().unique())
# model_number_models = set(model_number_db["ModelName"].str.lower().unique())
# missing_models = hyundai_models - model_number_models
# print("Models in hyundai_raw_df_filtered but not in model_number_db:")
# print(missing_models)
 

In [38]:

model_number_db[
                (1==1)
                & (model_number_db["ModelYear"]==2024)
                &(model_number_db["ModelName"].str.contains("G80"))
                # & (model_number_db["engine_type"].str.contains("ele"))
                ][["ModelName", "ModelYear", "TrimName", "Description", "ModelNumber", "engine_type"]]
 

,ModelName,ModelYear,TrimName,Description,ModelNumber,engine_type
467,Electrified G80,2024,Prestige,Awd,G8ES4ZE1GP00,electric
473,G80,2024,2.5T Advanced,2.5t Advanced Awd,G8CS4K2DGA00,2.5t
474,G80,2024,3.5T Sport Plus,3.5t Sport Plus Awd,G8CS4K3BGSAU,3.5t


In [39]:

# model_number_db[model_number_db["ModelName"]=="GV70"].value_counts(by=["ModelYear"])

model_number_db.loc[
    model_number_db["ModelName"]=="GV70",
    "ModelYear"
    ].value_counts()
 

ModelYear
2022    12
2023     6
2024     6
2027     6
2025     5
2026     5
Name: count, dtype: int64

In [40]:
model_number_db.columns

Index(['Description', 'Drivetrain', 'Manufacturer', 'ModelName', 'ModelNumber',
       'ModelYear', 'Package', 'PassDoors', 'TrimName', 'engine_type'],
      dtype='str')

In [41]:
make_df_genesis = model_number_db[model_number_db["Manufacturer"].str.contains("Genesis", case=False)]
make_df_hyundai = model_number_db[model_number_db["Manufacturer"].str.contains("Hyundai", case=False)]

In [42]:

make_df_genesis[make_df_genesis["TrimName"].isna()]


,Description,Drivetrain,Manufacturer,ModelName,ModelNumber,ModelYear,Package,PassDoors,TrimName,engine_type
528,Awd *ltd Avail*,ALL_WHEEL_DRIVE,GENESIS,GV60,V6EW5ZE2GW00,2026,481701,4,NaN,NaN


In [43]:
make_df_hyundai.columns

Index(['Description', 'Drivetrain', 'Manufacturer', 'ModelName', 'ModelNumber',
       'ModelYear', 'Package', 'PassDoors', 'TrimName', 'engine_type'],
      dtype='str')

In [44]:

make_df_hyundai[
    (make_df_hyundai["ModelYear"]==2026)
    # (1==1)
    &(make_df_hyundai["ModelName"].str.contains("ioniq 5", case=False, na=False))
    # &(make_df_hyundai["ModelName"].str.contains("fe", case=False, na=False))
    # & (make_df_hyundai["Description"].str.contains("XRT", case=False, na=False))
    # & (make_df_hyundai["TrimName"].str.contains("XRT", case=False, na=False))
    # & (make_df_hyundai["engine_type"].str.contains("Hybrid", case=False, na=False))
    # & (make_df_hyundai["Package"]=="KE00")
    ][["ModelYear","ModelNumber", "ModelName", "TrimName", "Description", "engine_type"]]
 

,ModelYear,ModelNumber,ModelName,TrimName,Description,engine_type
317,2026,I5EW5ZE4PRLR,IONIQ 5,Preferred,Preferred Rwd Long Range,NaN
318,2026,I5EW5ZE2PRLR,IONIQ 5,Preferred,Preferred Awd Long Range,NaN
319,2026,I5EW5ZE2PRUP,IONIQ 5,Preferred,Preferred Awd Long Range W/ultimate Package,NaN


In [45]:

make = "Hyundai"
year = 2026
keywords =  ['tucson', 'xrt']


search_models_by_description(make, year, keywords)
 

,Description,Drivetrain,Manufacturer,ModelName,ModelNumber,ModelYear,Package,PassDoors,TrimName,engine_type


In [46]:

year = 2026
Manufacturer = "hyundai"
ModelName = "elantra"
trimName  = 'TCR'

ModelName = "ioniq 5"
trimName  = 'n'

ModelName = "santa cruz"
trimName  = 'pref'

ModelName = "santa cruz"
trimName  = 'ult'

ModelName = "santa cruz"
trimName  = 'xrt'

ModelName = "santa fe"
trimName  = 'calli'

ModelName = "santa fe"
trimName  = 'lux'

ModelName = "santa fe"
trimName  = 'xrt'

ModelName = "tucson"
trimName  = 'xrt'




model_number_db[
    (1==1)
    & (model_number_db["ModelYear"] == year)
    & (model_number_db["Manufacturer"].str.contains(Manufacturer, case=False))
        & (model_number_db["ModelName"].str.contains(ModelName, case=False, na=False))
        & (model_number_db["TrimName"].str.contains(trimName, case=False, na=False))

    ][["ModelYear","Description", "ModelNumber", "ModelName", "TrimName", "Package", "engine_type","Manufacturer"]]


,ModelYear,Description,ModelNumber,ModelName,TrimName,Package,engine_type,Manufacturer


## Mitsubishi

In [47]:
# mitsubishi_df=(
#     mitsubishi_df[
# )

In [49]:

year = 2026
# Manufacturer = "Genesis"
ModelName = 'outlander'
trim = 'gt'
# Description = 'premium'

# mitsubishi_df[
#     (1==1)
#     & (mitsubishi_df["ModelYear"] == year)
#     # & (mitsubishi_df["Manufacturer"].str.contains(Manufacturer, case=False))
#     & (mitsubishi_df["ModelName"].str.contains(ModelName, case=False, na=False))
#     & (mitsubishi_df["TrimName"].str.contains(trim, case=False, na=False))
#     # & (mitsubishi_df["Description"].str.contains(Description, case=False, na=False))
#     ][["ModelYear", "ModelNumber", "ModelName", "TrimName", "Description", "Package", "engine_type","Manufacturer"]]

In [50]:
make = "mitsubishi"
year = 2026
keywords = ['outlander', 'gt', 'phev']


search_models_by_description(make, year, keywords)

,Description,Drivetrain,Manufacturer,ModelName,ModelNumber,ModelYear,Package,PassDoors,TrimName,engine_type
937,Gt Awc,ALL_WHEEL_DRIVE,MITSUBISHI,Outlander Plug-In Hybrid,COEV-X,2026,483111,4,GT,plug-in hybrid
938,Gt Noir Awc,ALL_WHEEL_DRIVE,MITSUBISHI,Outlander Plug-In Hybrid,COEV-N,2026,483112,4,GT NOIR,plug-in hybrid


## Genesis


In [51]:

Manufacturer = "Genesis"
# Description = 'premium'

year = 2024
ModelName = 'g70'
# trim = '2.0'
# Description = 'prestige'


year = 2024
ModelName = 'GV70'
Description = 'prestige' # Sames issue: Gv70 is labeled as Electrified GV70 in the db, need to find a way to handle it. 

year = 2025
ModelName = 'g70'
trim = '2.0'  # Missing in the DB

year = 2025
ModelName = 'gv80'
trim = '2.5t'  # Records in the db, but it's 

keywords = ['gv80', '2.5t', 'advanced', 'tech', 'pkg', '5p'] # Need to find out why this search is ot working for all the package items. This is happening for all the gv80 with tech package 


year = 2025
ModelName = 'gv80'
trim = '3.5t'  # his fails but the data is there. Ideas that Iam thinking,the 1) search keywords does not include EV, which fails the search condition to identify EV, 2) The keyword "coupe" is in Model name not the description 


year = 2026
ModelName = 'gv60'
trim = 'Performance'  # Missing data

year = 2026
ModelName = 'gv70'
trim = 'Advanced'  # Electrified issue


year = 2026
keywords = ['gv80', 'coupe', '3.5t', 'e-sc'] # this model /year is electric by default, and it has the coupe component in the model name, not the description. So the search is failing to find it. Same as ['gv80', 'coupe', '3.5t', 'e-sc', 'prestige', 'black'] 2026.



# year = 2027
ModelName = 'gv70'
# trim = 'prestige'  # Missing data

# year = 2027
# ModelName = 'gv70'
# trim = 'prestige'  # Electrified issue

# year = 2027
# ModelName = 'gv80'
# trim = 'advanced'  # Electrified issue


# year = 2027
# keywords = ['gv80', '2.5t', 'advanced', 'tech'] # this is available in the db but I am not sure why it's not being found. I will need to check the data in the db and see if it's there.


make_df_genesis[
    (1==1)
    & (make_df_genesis["ModelYear"] == year)
    & (make_df_genesis["Manufacturer"].str.contains(Manufacturer, case=False))
    & (make_df_genesis["ModelName"].str.contains(ModelName, case=False, na=False))
    & (make_df_genesis["TrimName"].str.contains(trim, case=False, na=False))
    # & (make_df_genesis["Description"].str.contains(Description, case=False, na=False))
    ][["ModelYear", "ModelNumber", "ModelName", "TrimName", "Description", "Package", "engine_type","Manufacturer"]]
 

,ModelYear,ModelNumber,ModelName,TrimName,Description,Package,engine_type,Manufacturer
515,2026,V7EW5ZE1GA00,Electrified GV70,Advanced,Advanced Awd,481478,electric,GENESIS
531,2026,V7CW5K2DGA00,GV70,2.5T Advanced,2.5t Advanced Awd,470386,2.5t,GENESIS
532,2026,V7CW5K2DGA55,GV70,2.5T Advanced Technology Package,2.5t Advanced Technology Package Awd,470387,2.5t,GENESIS


In [52]:

make = "genesis"
year = 2026
keywords = ['g70', '2.0t', 'advanced']
keywords = ['g80', 'ev', 'prestige']

keywords = ['g70', '2.0t', 'advanced']

keywords = ['gv80', '2.5t', 'advanced', 'tech', 'pkg', '5p'] # Need to find out why this search is ot working for all the package items


keywords = ['gv80', 'coupe', '3.5t']

keywords = ['gv70', 'ev', 'advanced']

year = 2026
keywords = ['gv80', 'coupe', '3.5t', 'e-sc']

keywords = ['gv60', 'magma']

keywords =  ['gv70', 'ev', 'prestige']

year = 2027
keywords = ['gv80', '2.5t', 'advanced', 'tech']

search_models_by_description(make, year, keywords)
 

,Description,Drivetrain,Manufacturer,ModelName,ModelNumber,ModelYear,Package,PassDoors,TrimName,engine_type
556,2.5t Advanced With-Technology Package Awd,ALL_WHEEL_DRIVE,GENESIS,GV80,V8CW7K2DGA55,2027,488631,4,2.5T Advanced w/Tech Pkg,2.5t


In [53]:
honda_df = model_number_db[model_number_db["Manufacturer"]=="HONDA"]


In [54]:

honda_df["Description"].value_counts()
 

Description
Sport AWD                          21
EX-L AWD                           16
Touring AWD                        16
TrailSport AWD                     14
Black Edition AWD                  13
Sport CVT                          12
Manual                              9
Touring CVT                         6
LX CVT                              6
EX CVT                              5
LX AWD                              5
LX 2WD                              5
Touring Auto                        5
Sport eCVT                          5
LX AWD CVT                          4
Sport AWD CVT                       4
LX 2WD CVT                          4
Black Edition Auto                  4
Sport Touring eCVT                  4
SE CVT                              3
Sport Touring Manual                3
Sport Touring CVT                   3
EX-L Navi AWD CVT                   3
Touring eCVT                        3
EX AWD                              3
LX Manual                           2


In [62]:

honda_df[
    honda_df["ModelName"].str.contains("CR-V")
    # honda_df["ModelName"].str.contains("CR-V")
    & 
    honda_df["Description"].str.contains("Sport")
    # & honda_df["ModelYear"].isin([2026])
]


,Description,Drivetrain,Manufacturer,ModelName,ModelNumber,ModelYear,Package,PassDoors,TrimName,engine_type
971,Sport AWD,ALL_WHEEL_DRIVE,HONDA,CR-V,RW2H4NJS,2022,425081,4,Sport,NaN
1016,Sport AWD,ALL_WHEEL_DRIVE,HONDA,CR-V,RS4H5PJS,2023,433353,4,Sport,NaN
1021,Sport-B AWD,ALL_WHEEL_DRIVE,HONDA,CR-V,RS4H5PJSX,2023,438189,4,Sport-B,NaN
1055,Sport AWD,ALL_WHEEL_DRIVE,HONDA,CR-V,RS4H5RJS,2024,443586,4,Sport,NaN
1094,Sport AWD,ALL_WHEEL_DRIVE,HONDA,CR-V,RS4H5SJS,2025,458858,4,Sport,NaN
1134,Sport AWD,ALL_WHEEL_DRIVE,HONDA,CR-V,RS4H5TJS,2026,477735,4,Sport,NaN
1138,Sport AWD,ALL_WHEEL_DRIVE,HONDA,CR-V Hybrid,RS6H5TJGX,2026,477739,4,Sport,NaN
1139,TrailSport AWD,ALL_WHEEL_DRIVE,HONDA,CR-V Hybrid,RS6H6TJG,2026,477740,4,TrailSport,NaN


In [63]:
honda_df.columns

Index(['Description', 'Drivetrain', 'Manufacturer', 'ModelName', 'ModelNumber',
       'ModelYear', 'Package', 'PassDoors', 'TrimName', 'engine_type'],
      dtype='str')